#What is tip.json?

##tip.json contains short suggestions, recommendations, or quick comments written by users for businesses.

##Step 1: Import Required Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import explode, split
from pyspark.sql.functions import count, when, col
from pyspark.sql.functions import year, month, quarter, dayofweek, hour, dayofmonth
import seaborn as sns
import matplotlib.pyplot as plt


##Step 2: List Available Files in S3

In [0]:
files = dbutils.fs.ls("s3://yelpdatasetsmvita/yelp_dataset/")
display(files)

path,name,size,modificationTime
s3://yelpdatasetsmvita/yelp_dataset/Dataset_User_Agreement.pdf,Dataset_User_Agreement.pdf,80358,1782571405000
s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_business.json,yelp_academic_dataset_business.json,118863795,1782571405000
s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_checkin.json,yelp_academic_dataset_checkin.json,286958945,1782571405000
s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_review.json,yelp_academic_dataset_review.json,5341868833,1782571405000
s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_tip.json,yelp_academic_dataset_tip.json,180604475,1782571405000
s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_user.json,yelp_academic_dataset_user.json,3363329011,1782571405000


#Bronze Layer:- Raw check-in data ingested from S3 without any transformations.


##Step 3: Load Tip Dataset

In [0]:
## Load tip dataset from S3 bucket
tip_df = spark.read.json(
    "s3://yelpdatasetsmvita/yelp_dataset/yelp_academic_dataset_tip.json"
)

Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xffb87068a900>> (for post_run_cell), with arguments args (<ExecutionResult object at ffb7ecd1b2c0, execution_count=8 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at ffb7ed0d6600, raw_cell="## Load tip dataset from S3 bucket
tip_df = spark..." store_history=True silent=False shell_futures=True cell_id=5891831125585766> result=None>,),kwargs {}:


In [0]:
#Preview Dataset
display(tip_df)

#DATA UNDERSTANDING

##Step 4: Understand Data Structure

In [0]:
# Display schema to understand dataset structure
tip_df.printSchema()

##Step 5: Rows & Columns

In [0]:
print("Rows:", tip_df.count())
print("Columns:", len(tip_df.columns))
tip_df.columns

#Silver Layer:- Data cleaning, timestamp conversion, and feature engineering.

#DATA QUALITY

##Step 6: Null Values

In [0]:
null_df = tip_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in tip_df.columns
])

display(null_df)

##Step 7: Removal of Duplicate Records

In [0]:
# Duplicate records
duplicate_count = tip_df.count() - tip_df.dropDuplicates().count()

print("Duplicate Records:", duplicate_count)

In [0]:
tip_df = tip_df.dropDuplicates()
tip_df.count()

##Step 8: Missing User IDs
###A tip without a user cannot be linked to customer behavior analysis.

In [0]:
missing_user_ids = tip_df.filter(
    col("user_id").isNull()).count()

print("Missing User IDs:", missing_user_ids)

##Step 9: Missing Business IDs
###A tip without a business cannot be associated with any business.

In [0]:
missing_business_ids =tip_df.filter(col("business_id").isNull()).count()

print("Missing Business IDs:", missing_business_ids)

##Step 10: Check Empty Strings(Especially for text)
###You should check both NULL values and empty strings.

In [0]:
empty_text = tip_df.filter(col("text") == "").count()
print("Empty Tips:", empty_text)

##Step 11: Invalid Timestamp Records

In [0]:
#First convert the date column to timestamp:
tip_df = tip_df.withColumn(
    "tip_timestamp",
    to_timestamp("date")
)


In [0]:
#Now check records that failed conversion:

invalid_dates = tip_df.filter(
    col("tip_timestamp").isNull() &
    col("date").isNotNull()
).count()

print("Invalid Dates:", invalid_dates)

##Step 12: Data Type Conversion
###Convert Date into Timestamp

In [0]:
tip_df = tip_df.withColumn(
    "tip_timestamp",
    to_timestamp("date")
)
tip_df.printSchema()

##Step 13: Create tip_clean

In [0]:
tip_clean = (
    tip_df
    .dropDuplicates()
    .filter(col("user_id").isNotNull())
    .filter(col("business_id").isNotNull())
    .filter(trim(col("text")) != "")
    .filter(col("date").isNotNull())
)

In [0]:
tip_clean.count()
tip_clean.printSchema()

#FEATURE ENGINEERING

##Step 14: Temporal Features

In [0]:
from pyspark.sql.functions import*

tip_final = (
    tip_clean
    .withColumn(
        "year",
        year(col("tip_timestamp"))
    )
    .withColumn(
        "month",
        month(col("tip_timestamp"))
    )
    .withColumn(
        "weekday",
        date_format(
            col("tip_timestamp"),
            "EEEE"
        )
    )
    .withColumn(
        "hour",
        hour(col("tip_timestamp"))
    )
    .withColumn(
        "quarter", 
        quarter(col("tip_timestamp"))
    )
    
    .withColumn(
        "tip_length",
        length(col("text"))
    )
    .withColumn(
        "word_count",
        size(
            split(
                col("text"),
                " "
            )
        )
    )
)

# Cache since tip_final is reused across many downstream transformations/actions
tip_final = tip_final.cache()

display(tip_final)

#TEXT FEATURES

##Step 16: Tip Category
###Short / Medium / Long Tips

In [0]:
tip_final = tip_final.withColumn(
    "tip_category",
    when(col("word_count") <= 5, "Short")
    .when(col("word_count") <= 15, "Medium")
    .otherwise("Long")
)

##Step 17: Active vs Casual Users

In [0]:
user_tip_counts = (
    tip_final
    .groupBy("user_id")
    .count()
    .withColumnRenamed("count", "total_tips")
)

user_segments = (
    user_tip_counts
    .withColumn(
        "user_type",
        when(col("total_tips") >= 10, "Active User")
        .otherwise("Casual User")
    )
)

display(
    user_segments
    .groupBy("user_type")
    .count()
)

#Plot the data using Matplotlib
plot_data = user_segments.groupBy("user_type").count().toPandas()
plt.figure(figsize=(6, 6))
plt.pie(plot_data["count"], labels=plot_data["user_type"], autopct="%1.1f%%",
        colors=["#2E74B5", "#DCE6F1"])
plt.title("Active vs Casual Users")
plt.show()

##Step 18: Highly Engaged Businesses

In [0]:
business_tip_counts = (
    tip_final
    .groupBy("business_id")
    .count()
    .withColumnRenamed("count", "total_tips")
)

engaged_businesses = (
    business_tip_counts
    .withColumn(
        "engagement_level",
        when(col("total_tips") >= 20, "Highly Engaged")
        .otherwise("Normal Engagement")
    )
)

display(
    engaged_businesses
    .groupBy("engagement_level")
    .count()
)

#Plot the data using Matplotlib
plot_data = engaged_businesses.groupBy("engagement_level").count().toPandas()
plt.figure(figsize=(6, 6))
plt.pie(plot_data["count"], labels=plot_data["engagement_level"], autopct="%1.1f%%",
        colors=["#2E74B5", "#DCE6F1"])
plt.title("Highly Engaged vs Normal Businesses")
plt.show()

##Step 19: Unique Users 

In [0]:
tip_final.select(
    countDistinct("user_id")
).show()

##Step 20: Unique Businesses

In [0]:
tip_final.select(
    countDistinct("business_id")
    ).show()

##Step 21: Average Tip Length

In [0]:
tip_final.select(
    avg("tip_length")
).show()

##Step 22: Median Tip Length

In [0]:
tip_final.approxQuantile(
    "tip_length",
    [0.5],
    0.01
)

##Step 23: Average Words Per Tip

In [0]:
tip_final.select(
    avg("word_count")
).show()

##Step 24: Dataset Date Range

In [0]:
tip_final.select(
    min("date").alias("start_date"),
    max("date").alias("end_date")
).show()

#CUSTOMER ENGAGEMENT ANALYSIS

##Step 25: Tips by Year

In [0]:
display(
    tip_final
    .groupBy("year")
    .count()
    .orderBy("year")
)
pdf = tip_final.groupBy("year").count().orderBy("year").toPandas()
plt.figure(figsize=(8, 5))
plt.plot(pdf["year"], pdf["count"], marker="o", color="#2E74B5")
plt.title("Yearly Tip Trend")
plt.xlabel("Year")
plt.ylabel("Total Tips")
plt.grid(alpha=0.3)
plt.show()

##Step 26: Tips by Month

In [0]:
display(
    tip_final
    .groupBy("month")
    .count()
    .orderBy("month")
)


#Plot the data using Matplotlib
plot_data = tip_final.groupBy("month").count().orderBy("month").toPandas()
plt.figure(figsize=(8, 5))
plt.plot(plot_data["month"], plot_data["count"], marker="o", color="#2E74B5")
plt.title("Monthly Tip Trend")
plt.xlabel("Month")
plt.ylabel("Total Tips")
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.show()

##Step 27: Tips by Weekday

In [0]:
display(
    tip_final
    .groupBy("weekday")
    .count()
)

#Plot the data using Matplotlib
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plot_data = tip_final.groupBy("weekday").count().toPandas()
plot_data["weekday"] = plot_data["weekday"].astype("category").cat.set_categories(order)
plot_data = plot_data.sort_values("weekday")
plt.figure(figsize=(8, 5))
plt.bar(plot_data["weekday"], plot_data["count"], color="#2E74B5")
plt.title("Tips by Weekday")
plt.ylabel("Total Tips")
plt.xticks(rotation=30)
plt.show()

##Step 28: Tips by Hour

In [0]:
display(
    tip_final
    .groupBy("hour")
    .count()
    .orderBy("hour")
)

#Plot the data using Matplotlib
plot_data = tip_final.groupBy("hour").count().orderBy("hour").toPandas()
plt.figure(figsize=(10, 5))
plt.bar(plot_data["hour"], plot_data["count"], color="#2E74B5")
plt.title("Tips by Hour of Day")
plt.xlabel("Hour (24h)")
plt.ylabel("Total Tips")
plt.xticks(range(0, 24))
plt.show()

##Step 29: Peak Engagement Periods

In [0]:
display(
    tip_final
    .groupBy("weekday", "hour")
    .count()
    .orderBy(col("count").desc())
)

# Heatmap — Weekday x Hour (peak engagement window)
plot_data = tip_final.groupBy("weekday", "hour").count().toPandas()
pivot = plot_data.pivot(index="weekday", columns="hour", values="count").reindex(order)
plt.figure(figsize=(14, 6))
sns.heatmap(pivot, cmap="Blues", linewidths=0.3)
plt.title("Tip Engagement Heatmap: Weekday x Hour")
plt.xlabel("Hour")
plt.ylabel("Weekday")
plt.show()

#USER BEHAVIOR ANALYSIS

##Step 30: Top Most 20 Active Users

In [0]:
display(
    tip_final
    .groupBy("user_id")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)

#Plot the data using Matplotlib
plot_data = (
    tip_final
    .groupBy("user_id")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
    .toPandas()
)
plt.figure(figsize=(9, 8))
plt.barh(plot_data["user_id"], plot_data["count"], color="#2E74B5")
plt.title("Top 20 Active Users by Tip Count")
plt.xlabel("Total Tips")
plt.gca().invert_yaxis()
plt.show()

##Step 31: Average Tips Per User

In [0]:
tips_per_user = (
    tip_final
    .groupBy("user_id")
    .count()
)

tips_per_user.select(
    avg("count")
).show()

##Step 32: Distribution of Tips per Users

In [0]:
display(
    tips_per_user
    .groupBy("count")
    .count()
    .orderBy("count")
)

#Plot the data usibg Matplotlib
plot_data = tips_per_user.toPandas()
plt.figure(figsize=(9, 5))
plt.hist(plot_data["count"], bins=30, color="#2E74B5", edgecolor="white")
plt.title("Distribution of Tips per User")
plt.xlabel("Number of Tips")
plt.ylabel("Number of Users")
plt.yscale("log")   # counts are heavily skewed, log scale makes the tail visible
plt.show()

##Step 33: Power Users
### Users with more than 50 tips:

In [0]:
power_users = (
    tips_per_user
    .filter(col("count") > 50)
)

display(power_users)

power_user_count = power_users.count()
total_user_count = tips_per_user.count()
print(f"Power Users (>50 tips): {power_user_count} out of {total_user_count} "
      f"({(power_user_count/total_user_count)*100:.2f}%)")

# TOP CONTRIBUTIONS

## Step 34: Top 1%, 5%, 10%

In [0]:
############# TOP 1% ############################

threshold_1 = tips_per_user.approxQuantile(
    "count",
    [0.99],
    0.01
)[0]

top_1_users = (
    tips_per_user
    .filter(col("count") >= threshold_1)
)
display(top_1_users)

############# TOP 5% ############################
threshold_5 = tips_per_user.approxQuantile(
    "count",
    [0.95],
    0.01
)[0]

top_5_users = (
    tips_per_user
    .filter(col("count") >= threshold_5)
)
display(top_5_users)

############# TOP 10% ############################
threshold_10 = tips_per_user.approxQuantile(
    "count",
    [0.90],
    0.01
)[0]

top_10_users = (
    tips_per_user
    .filter(col("count") >= threshold_10)
)
display(top_10_users)

#BUSINESS ENGAGEMENT ANALYSIS

##Step 35: Top 20 Businesses by Tip Count

In [0]:
display(
    tip_final
    .groupBy("business_id")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
)


#Plot the data using Matplotlib
plot_data = (
    tip_final
    .groupBy("business_id")
    .count()
    .orderBy(col("count").desc())
    .limit(20)
    .toPandas()
)
plt.figure(figsize=(9, 8))
plt.barh(plot_data["business_id"], plot_data["count"], color="#2E74B5")
plt.title("Top 20 Businesses by Tip Count")
plt.xlabel("Total Tips")
plt.gca().invert_yaxis()
plt.show()

##Step 36: Bottom 20 Businesses by Tip Count

In [0]:
display(
    tip_final
    .groupBy("business_id")
    .count()
    .orderBy(col("count"))
    .limit(20)
)

##Step 37: Average Tips Per Business

In [0]:
business_counts = (
    tip_final
    .groupBy("business_id")
    .count()
)

business_counts.select(
    avg("count")
).show()

##Step 38: Business Engagement Distribution

In [0]:

display(
    business_counts
    .groupBy("count")
    .count()
    .orderBy("count")
)

#Plot the data using Matplotlib
plot_data = business_counts.toPandas()
plt.figure(figsize=(9, 5))
plt.hist(plot_data["count"], bins=30, color="#2E74B5", edgecolor="white")
plt.title("Business Engagement Distribution (Tips per Business)")
plt.xlabel("Number of Tips")
plt.ylabel("Number of Businesses")
plt.yscale("log")
plt.show()

##Step 39: Highly Engaged Businesses

In [0]:
high_engagement_businesses = (
    business_counts
    .filter(col("count") >= 20)
)

display(high_engagement_businesses)

#TEXT ANALYTICS

##Step 40: Tip Length Distribution

In [0]:
display(
    tip_final
    .groupBy("tip_category")
    .count()
)

#Plot the data using Matplotlib
plot_data = tip_final.groupBy("tip_category").count().toPandas()
plt.figure(figsize=(6, 6))
plt.pie(plot_data["count"], labels=plot_data["tip_category"], autopct="%1.1f%%",
        colors=["#2E74B5", "#5B9BD5", "#DCE6F1"])
plt.title("Tip Length Category Distribution")
plt.show()

##Step 41: Top 20 Most Frequent Words

In [0]:
words_df = (
    tip_final
    .select(
        explode(
            split(
                lower(col("text")),
                " "
            )
        ).alias("word")
    )
)

display(
    words_df
    .groupBy("word")
    .count()
    .orderBy(col("count").desc())
    .limit(50)
)


#Plot the data using Matplotlib
plot_data = (
    words_df.groupBy("word").count()
    .orderBy(col("count").desc())
    .limit(20)
    .toPandas()
)
plt.figure(figsize=(9, 8))
plt.barh(plot_data["word"], plot_data["count"], color="#2E74B5")
plt.title("Top 20 Most Frequent Words in Tips")
plt.xlabel("Frequency")
plt.gca().invert_yaxis()
plt.show()


##Step 42: Recommendation Keywords Frequency

In [0]:
recommendation_keywords = [
    "recommend",
    "try",
    "must",
    "best",
    "favorite"
]

display(
    words_df.filter(
        col("word").isin(recommendation_keywords)
    )
    .groupBy("word")
    .count()
)


#Plot the data using Matplotlib
plot_data = (
    words_df.filter(col("word").isin(recommendation_keywords))
    .groupBy("word").count()
    .toPandas()
)
plt.figure(figsize=(7, 5))
plt.bar(plot_data["word"], plot_data["count"], color="#2E74B5")
plt.title("Recommendation Keyword Frequency")
plt.ylabel("Frequency")
plt.show()

##Step 43: Experience Keywords Frequency



In [0]:
experience_keywords = [
    "service",
    "food",
    "staff",
    "friendly",
    "clean"
]

display(
    words_df.filter(
        col("word").isin(experience_keywords)
    )
    .groupBy("word")
    .count()
)

#Plot the data using Matplotlib
plot_data = (
    words_df.filter(col("word").isin(experience_keywords))
    .groupBy("word").count()
    .toPandas()
)
plt.figure(figsize=(7, 5))
plt.bar(plot_data["word"], plot_data["count"], color="#2E74B5")
plt.title("Experience Keyword Frequency")
plt.ylabel("Frequency")
plt.show()


#BUSINESS INSIGHTS

- ### Customer engagement peaks during specific hours.
- ### Most customers leave short feedback messages.
- ### A small percentage of users generate a large proportion of tips.
- ### Certain businesses attract significantly higher engagement.
- ### Engagement varies seasonally and annually.

# BUSINESS OUTCOMES

- ### Target loyal customers with retention programs.
- ### Identify businesses suitable for promotional campaigns.
- ### Use customer language in marketing initiatives.
- ### Monitor customer engagement trends over time.
- ### Detect changes in customer sentiment and experience.

#Final Flow of Code — TIP.JSON

##1. Import Required Libraries
            ↓

#       Bronze Layer
##2. Load Raw tip.json from S3
            ↓

#     Silver Layer
##3. Dataset Understanding
- ###Schema
- ###Row Count
- ###Column Count
- ###Sample Records
- ###Data Types

            ↓

##4. Data Quality Validation
- ###Missing User IDs
- ###Missing Business IDs
- ###Empty Tip Text
- ###Duplicate Records

            ↓

##5. Timestamp Conversion
###Convert date to TimestampType

            ↓


##6. Feature Engineering
###Extract:
- ###Year
- ###Quarter
- ###Month
- ###Weekday
- ###Hour
- ###Tip Length
- ###Word Count

##Create:

- ###Tip Category
- ###Active vs Casual Users
- ###Highly Engaged Businesses
            ↓
##7. Dataset Summary Statistics
- ###Total Tips
- ###Unique Users
- ###Unique Businesses
- ###Average Tip Length
- ###Median Tip Length
- ###Average Words Per Tip
- ###Dataset Date Range
            ↓

#Gold Layer

##8. Customer Engagement Analysis
- ###Tips by Year
- ###Tips by Month
- ###Tips by Weekday
- ###Tips by Hour
- ###Peak Engagement Periods

            ↓

##9. User Behavior Analysis
- ###Most Active Users
- ###Average Tips per User
- ###User Contribution Distribution
- ###Power Users
- ###Top 1%, 5%, and 10% Contributors
            
            ↓

##10. Business Engagement Analysis
- ###Businesses Receiving Most Tips
- ###Businesses Receiving Least Tips
- ###Average Tips per Business
- ###Business Engagement Distribution
- ###Highly Engaged Businesses

            ↓

##11. Text Analytics
- ###Tip Length Distribution
- ###Most Frequent Words
- ###Recommendation Keywords
- ###Experience Keywords

            ↓

##12. Business Insights

            ↓

##13. Business Outcomes